# 03_Data_Preparation

Notebook ini mendokumentasikan fase Data Preparation. Fokus pada pembersihan format harga, penanganan missing values, transformasi data produktivitas, dan persiapan dataset untuk pemodelan LSTM.

## Aktivasi Environment RAPIDS

Gunakan environment conda `rapids-24.10` sebelum menjalankan notebook ini. Contoh perintah:

```bash
conda activate rapids-24.10
```


## Tujuan Data Preparation

1. Membersihkan format harga dan mengonversi ke tipe numerik.
2. Menangani missing values dan nilai tanggal tidak valid.
3. Menyiapkan data produktivitas triwulanan sebagai fitur eksogen.
4. Menyimpan dataset hasil pra-pemrosesan ke folder `data/processed/`.

In [14]:
import pandas as pd
from pathlib import Path

raw_dir = Path('../data/raw')
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

price_path = raw_dir / 'DATASET HARGA KOMODITI.csv'
prod_path = raw_dir / 'produksi.csv'

price_df = pd.read_csv(price_path)
price_df.columns = [col.strip() for col in price_df.columns]
price_df['Komoditi'] = price_df['Komoditi'].astype(str).str.strip()
price_df['Tanggal'] = pd.to_datetime(price_df['Tanggal'], errors='coerce')

prod_df = pd.read_csv(prod_path)
prod_df['Komoditi'] = prod_df['Komoditi'].astype(str).str.strip()

price_df.head(3)

,Komoditi,Tanggal,Harga Petani,Harga Pengecer
0,Jagung Pipil Kering,2022-01-03,"Rp4,800.00","Rp5,000.00"
1,Jagung Pipil Kering,2022-01-04,"Rp4,800.00","Rp5,000.00"
2,Jagung Pipil Kering,2022-01-05,"Rp4,800.00","Rp5,000.00"


## Cleaning Format Harga

Kolom harga `Harga Petani` dan `Harga Pengecer` dibersihkan dari simbol mata uang dan koma, lalu dikonversi menjadi numerik.

In [15]:
price_df['Harga Petani'] = pd.to_numeric(price_df['Harga Petani'].astype(str).str.replace('[^0-9.]', '', regex=True), errors='coerce')
price_df['Harga Pengecer'] = pd.to_numeric(price_df['Harga Pengecer'].astype(str).str.replace('[^0-9.]', '', regex=True), errors='coerce')

print('Tipe data setelah pembersihan:')
print(price_df[['Harga Petani', 'Harga Pengecer']].dtypes)
print('\nContoh record setelah pembersihan:')
print(price_df.head(5).to_string(index=False))


Tipe data setelah pembersihan:
Harga Petani      float64
Harga Pengecer    float64
dtype: object

Contoh record setelah pembersihan:
           Komoditi    Tanggal  Harga Petani  Harga Pengecer
Jagung Pipil Kering 2022-01-03        4800.0          5000.0
Jagung Pipil Kering 2022-01-04        4800.0          5000.0
Jagung Pipil Kering 2022-01-05        4800.0          5000.0
Jagung Pipil Kering 2022-01-06        4800.0          5000.0
Jagung Pipil Kering 2022-01-07        4800.0          5000.0


## Menangani Missing Values dan Nilai Tanggal Tidak Valid

Terdapat satu baris tanggal tidak valid dan sejumlah missing values pada kolom harga. Harga diimputasi menggunakan forward fill per komoditas, kemudian backward fill untuk mengisi nilai awal yang kosong.

In [16]:
print('Jumlah baris sebelum pembersihan tanggal:', len(price_df))
invalid_dates = price_df['Tanggal'].isna().sum()
print('Nilai tanggal tidak valid:', invalid_dates)
price_df = price_df.dropna(subset=['Tanggal']).copy()
price_df = price_df.sort_values(['Komoditi', 'Tanggal']).reset_index(drop=True)

missing_before = price_df[['Harga Petani', 'Harga Pengecer']].isna().sum()
print('Missing sebelum imputasi:', missing_before.to_dict())

price_df[['Harga Petani', 'Harga Pengecer']] = price_df.groupby('Komoditi')[['Harga Petani', 'Harga Pengecer']].ffill().bfill()
missing_after = price_df[['Harga Petani', 'Harga Pengecer']].isna().sum()
print('Missing setelah imputasi:', missing_after.to_dict())


Jumlah baris sebelum pembersihan tanggal: 4258
Nilai tanggal tidak valid: 1
Missing sebelum imputasi: {'Harga Petani': 2157, 'Harga Pengecer': 2156}
Missing setelah imputasi: {'Harga Petani': 0, 'Harga Pengecer': 0}


## Transformasi Data Produktivitas

Data produktivitas disiapkan dengan mengonversi tahun dan triwulan menjadi periode kuartal. Terdapat pencocokan langsung untuk jagung dan kacang hijau. Untuk beras, data `padi total` tercatat sebagai proxy terhadap variasi harga beras.

In [17]:
prod_df['quarter_start'] = pd.to_datetime(prod_df['tahun'].astype(str) + 'Q' + prod_df['triwulan'].astype(str))
prod_df['quarter_start'] = prod_df['quarter_start'].dt.to_period('Q').dt.start_time
prod_df = prod_df.sort_values(['Komoditi', 'quarter_start']).reset_index(drop=True)
print(prod_df[['Komoditi', 'tahun', 'triwulan', 'quarter_start', 'Produktivitas']].head(10).to_string(index=False))


  Komoditi  tahun  triwulan quarter_start  Produktivitas
Padi sawah   2022         1    2022-01-01      56.593625
Padi sawah   2022         2    2022-04-01      54.279752
Padi sawah   2022         3    2022-07-01      55.240632
Padi sawah   2023         1    2023-01-01      55.977470
Padi sawah   2023         2    2023-04-01      58.797478
Padi sawah   2023         3    2023-07-01      54.176608
    jagung   2022         1    2022-01-01      73.898901
    jagung   2022         2    2022-04-01      72.218328
    jagung   2022         3    2022-07-01      68.690000
    jagung   2023         1    2023-01-01      71.010134


/tmp/ipykernel_28196/3554880535.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  prod_df['quarter_start'] = pd.to_datetime(prod_df['tahun'].astype(str) + 'Q' + prod_df['triwulan'].astype(str))


## Penyusunan Fitur Eksogen untuk Harga

Menetapkan mapping komoditas antara data harga dan data produktivitas. Pemetaan tersebut akan digunakan di tahap berikutnya untuk menggabungkan informasi produksi/produktifitas ke dalam dataset harga.

In [18]:
commodity_map = {
    'jagung': 'Jagung Pipil Kering',
    'kacang hijau': 'Kacang Hijau',
    'padi total': 'Beras Medium',
    'padi sawah': 'Beras Premium',
    'padi gogo': 'Beras Premium'
}
prod_df['Komoditi_Harga'] = prod_df['Komoditi'].str.lower().map(commodity_map)
print(prod_df[['Komoditi', 'Komoditi_Harga']].drop_duplicates().to_string(index=False))


    Komoditi      Komoditi_Harga
  Padi sawah       Beras Premium
      jagung Jagung Pipil Kering
kacang hijau        Kacang Hijau
   padi gogo       Beras Premium
  padi total        Beras Medium


## Penyimpanan Data yang Diproses

Setelah pembersihan dan transformasi awal, data disimpan ke folder `data/processed/` agar siap digunakan di fase modeling.

In [19]:
price_df.to_csv(processed_dir / 'price_cleaned.csv', index=False)
prod_df.to_csv(processed_dir / 'production_transformed.csv', index=False)
print('Data hasil pembersihan telah disimpan di data/processed/')


Data hasil pembersihan telah disimpan di data/processed/


## Insight Data Preparation

- Pembersihan format harga berhasil mengubah kolom `Harga Petani` dan `Harga Pengecer` menjadi tipe numerik.
- Satu nilai `Tanggal` invalid dibuang, sehingga seluruh baris harga memiliki informasi tanggal yang valid.
- Missing values harga diimputasi dengan forward fill dan backward fill per komoditas, yang sesuai untuk analisis time series berbasis LSTM.
- Data produktivitas triwulanan telah dikonversi ke `quarter_start` sehingga dapat digabungkan dengan data harga berdasarkan periode kuartal.
- Untuk beras, dataset produktivitas menggunakan proxy `padi total`/`padi sawah`; sementara jagung dan kacang hijau dapat langsung dipetakan.

Fase berikutnya adalah feature engineering untuk LSTM: membuat lag harga, indikator musiman, kalender libur, dan menyiapkan sequence time series untuk model univariat dan multivariat.